# Semana 3 — Entrenamiento y Selección del Mejor Modelo

Clasificador de rostros de 26 futbolistas argentinos (FIFA 2022).  
Fine-tuning de modelos preentrenados en PyTorch, comparación con mAP.

| # | Sección |
|---|---|
| 1 | Setup |
| 2 | Datos |
| 3 | Métricas |
| 4 | Funciones de entrenamiento |
| **A** | ResNet-50 — backbone congelado (baseline) |
| **B** | EfficientNet-B2 — fine-tuning parcial |
| **C** | MobileNetV3-Large — fine-tuning end-to-end |
| 5 | Comparación y selección del ganador |
| 6 | Evaluación en test |
| 7 | Guardado del modelo |

In [ ]:
# ── 1. Setup ─────────────────────────────────────────────────
!pip install -q torchmetrics gdown

In [ ]:
import json
import os
import random
import zipfile
from pathlib import Path

import gdown
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchmetrics.classification import (
    MulticlassAccuracy,
    MulticlassAveragePrecision,
    MulticlassConfusionMatrix,
    MulticlassF1Score,
)
from torchvision import models, transforms
from torchvision.models import (
    EfficientNet_B2_Weights,
    MobileNet_V3_Large_Weights,
    ResNet50_Weights,
)
import warnings
warnings.filterwarnings('ignore')

In [ ]:
SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU   : {torch.cuda.get_device_name(0)}')

In [ ]:
# ============================================================
# CONFIGURACIÓN GLOBAL
# ============================================================

DRIVE_ZIP_ID = '1bf4dkkJWgH02et9NfVVxJis1X4C9nLsu'

FACES_DIR   = '/content/FIFA_2022_ONLY_FACES'
DATA_DIR    = '/content/data'       # CSVs versionados en el repo

IMG_SIZE    = 224
BATCH_SIZE  = 32
NUM_WORKERS = 2
NUM_CLASSES = 26

In [ ]:
# ── 2. Datos ─────────────────────────────────────────────────

zip_path = '/content/FIFA_2022_ONLY_FACES.zip'

if not Path(FACES_DIR).exists():
    print('Descargando ZIP de rostros desde Drive...')
    gdown.download(f'https://drive.google.com/uc?id={DRIVE_ZIP_ID}', zip_path, quiet=False)
    print('Descomprimiendo...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('/content')
    print(f'Listo — rostros en: {FACES_DIR}')
else:
    print('ZIP ya descomprimido, omitiendo descarga.')

train_df = pd.read_csv(f'{DATA_DIR}/train.csv')
val_df   = pd.read_csv(f'{DATA_DIR}/val.csv')
test_df  = pd.read_csv(f'{DATA_DIR}/test.csv')

with open(f'{DATA_DIR}/label_to_idx.json') as f:
    label_to_idx = json.load(f)

idx_to_label = {v: k for k, v in label_to_idx.items()}

print(f'Train : {len(train_df)} imagenes')
print(f'Val   : {len(val_df)} imagenes')
print(f'Test  : {len(test_df)} imagenes')
print(f'Clases: {NUM_CLASSES}')

In [ ]:
class FaceDataset(Dataset):
    """
    Extiende la clase de semana 2 con dos flags adicionales:
    use_rotation y use_erasing, necesarios para el Exp C.
    """

    def __init__(self, dataframe, root_dir,
                 use_flip=False, use_color_jitter=False,
                 use_rotation=False, use_erasing=False):
        self.df       = dataframe.reset_index(drop=True)
        self.root_dir = Path(root_dir)

        steps = [transforms.Resize((IMG_SIZE, IMG_SIZE))]
        if use_flip:
            steps.append(transforms.RandomHorizontalFlip(p=0.5))
        if use_color_jitter:
            steps.append(transforms.ColorJitter(
                brightness=0.2, contrast=0.2, saturation=0.2))
        if use_rotation:
            steps.append(transforms.RandomRotation(degrees=10))
        steps += [
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                  std=[0.229, 0.224, 0.225]),
        ]
        # RandomErasing opera sobre tensor: va DESPUES de ToTensor
        if use_erasing:
            steps.append(transforms.RandomErasing(p=0.2, scale=(0.02, 0.15)))

        self.transform = transforms.Compose(steps)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(self.root_dir / row['image_path']).convert('RGB')
        return self.transform(img), int(row['label_idx'])


def build_dataloader(df, use_flip=False, use_color_jitter=False,
                     use_rotation=False, use_erasing=False, shuffle=False):
    ds = FaceDataset(df, FACES_DIR,
                     use_flip=use_flip, use_color_jitter=use_color_jitter,
                     use_rotation=use_rotation, use_erasing=use_erasing)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=True)

## 3. Métricas — mAP en clasificación

El **mAP** que vimos en la materia es la métrica estándar de detectores de objetos:  
mide cómo el detector *rankea* sus detecciones por score de confianza a distintos umbrales de IoU.

Para clasificación multiclase aplicamos el mismo principio con una adaptación:

1. Para cada una de las 26 clases, se plantea el problema como **one-vs-rest**  
   (¿es o no es el jugador $k$?).
2. Se usa la **probabilidad softmax** como score de confianza.
3. Se calcula el **Average Precision** de cada clase = área bajo la curva precision-recall.
4. El **mAP (macro)** = promedio de los 26 AP individuales:

$$\text{mAP} = \frac{1}{26} \sum_{k=1}^{26} \text{AP}_k$$

**Ventaja sobre accuracy:** captura la calidad del *ranking* de probabilidades, no solo el top-1.  
Si el modelo asigna 0.8 a la clase correcta versus 0.51, eso se refleja en el AP.  
Además, como algunas clases tienen menos de 40 imágenes de entrenamiento, el AP por clase  
captura el rendimiento individual antes de promediar — es una métrica más justa bajo desbalance.

```python
MulticlassAveragePrecision(num_classes=26, average='macro')
```

Junto con mAP reportamos **accuracy macro** y **F1-macro** como pide la consigna.

In [ ]:
# ── 4. Funciones de entrenamiento ────────────────────────────

def make_metrics():
    return {
        'map': MulticlassAveragePrecision(num_classes=NUM_CLASSES, average='macro').to(DEVICE),
        'acc': MulticlassAccuracy(num_classes=NUM_CLASSES, average='macro').to(DEVICE),
        'f1' : MulticlassF1Score(num_classes=NUM_CLASSES, average='macro').to(DEVICE),
    }


def train_one_epoch(model, loader, criterion, optimizer, batch_scheduler=None):
    model.train()
    running_loss = 0.0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        # OneCycleLR debe avanzarse por batch, no por epoca
        if batch_scheduler is not None:
            batch_scheduler.step()
        running_loss += loss.item() * imgs.size(0)
    return running_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, criterion, metrics):
    model.eval()
    for m in metrics.values():
        m.reset()
    running_loss = 0.0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        logits = model(imgs)
        running_loss += criterion(logits, labels).item() * imgs.size(0)
        probs = torch.softmax(logits, dim=1)
        for m in metrics.values():
            m.update(probs, labels)
    return running_loss / len(loader.dataset), {k: m.compute().item() for k, m in metrics.items()}


def run_experiment(name, model, train_loader, val_loader, criterion, optimizer, n_epochs,
                   epoch_scheduler=None, batch_scheduler=None, scheduler_metric='map'):
    metrics  = make_metrics()
    history  = {'train_loss': [], 'val_loss': [], 'val_map': [], 'val_acc': [], 'val_f1': []}
    best_map = 0.0
    best_wts = None

    for epoch in range(1, n_epochs + 1):
        train_loss      = train_one_epoch(model, train_loader, criterion, optimizer, batch_scheduler)
        val_loss, val_m = evaluate(model, val_loader, criterion, metrics)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_map'].append(val_m['map'])
        history['val_acc'].append(val_m['acc'])
        history['val_f1'].append(val_m['f1'])

        if epoch_scheduler is not None:
            if isinstance(epoch_scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                epoch_scheduler.step(val_m[scheduler_metric])
            else:
                epoch_scheduler.step()

        if val_m['map'] > best_map:
            best_map = val_m['map']
            best_wts = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        if epoch % 5 == 0 or epoch == n_epochs:
            print(f'[{name}] Ep {epoch:02d}/{n_epochs} | '
                  f'Loss tr={train_loss:.4f} val={val_loss:.4f} | '
                  f"mAP={val_m['map']:.4f} Acc={val_m['acc']:.4f} F1={val_m['f1']:.4f}")

    model.load_state_dict(best_wts)
    return model, history, best_map


def plot_curves(history, name):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

    ax1.plot(history['train_loss'], label='train')
    ax1.plot(history['val_loss'],   label='val', linestyle='--')
    ax1.set_title(f'{name} — Perdida')
    ax1.set_xlabel('Epoca')
    ax1.legend()
    ax1.grid(alpha=0.3)

    ax2.plot(history['val_map'], label='mAP')
    ax2.plot(history['val_acc'], label='Accuracy')
    ax2.plot(history['val_f1'],  label='F1-macro')
    ax2.set_title(f'{name} — Metricas (val)')
    ax2.set_xlabel('Epoca')
    ax2.legend()
    ax2.grid(alpha=0.3)

    plt.suptitle(name, fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

---
## Experimento A — ResNet-50 | Solo cabeza (baseline)

### Por qué ResNet-50

ResNet-50 es el punto de partida natural para cualquier benchmark de fine-tuning porque:
- Su comportamiento bajo distintas estrategias está documentado extensamente → sabemos qué esperar.
- Los pesos `IMAGENET1K_V2` se obtuvieron con recetas modernas de entrenamiento  
  (MixUp, CutMix, label smoothing) → son una base más fuerte que los V1 originales.
- La arquitectura residual evita gradientes desvanecientes → es estable durante el fine-tuning.

### Por qué congelar todo el backbone

Con ~737 imágenes de entrenamiento para 26 clases (~28 por clase), el backbone completo  
tiene **25 millones de parámetros**. Entrenarlos todos causaría sobreajuste severo.  
Al congelar el backbone y entrenar únicamente la cabeza (`fc`: 2048 → 26),  
reducimos el espacio de optimización a **~53 000 parámetros**.

Este experimento responde a la pregunta clave:  
*¿son las features genéricas de ImageNet suficientes para distinguir entre estos 26 jugadores?*  
Si el mAP ya es aceptable, no vale la pena el riesgo de overfitting de descongelar más capas.

### Por qué sin augmentaciones

Intencionalmente no se usan augmentaciones en train para **aislar el efecto de la estrategia  
de fine-tuning** del efecto de la augmentación. Esto hace la comparación A vs. B vs. C más limpia.

### Por qué Adam + ReduceLROnPlateau

Con un único grupo de parámetros (la cabeza), Adam converge más rápido que SGD.  
`ReduceLROnPlateau(patience=3)` divide el LR a la mitad cuando el mAP en validación  
no mejora en 3 épocas consecutivas, evitando que el optimizer oscile alrededor del mínimo.

In [ ]:
set_seed()

# ── Modelo ──────────────────────────────────────────────────
model_a = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)

for param in model_a.parameters():
    param.requires_grad = False

model_a.fc = nn.Linear(model_a.fc.in_features, NUM_CLASSES)
model_a    = model_a.to(DEVICE)

n_train = sum(p.numel() for p in model_a.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model_a.parameters())
print(f'Parametros entrenables: {n_train:,} / {n_total:,}')

# ── DataLoaders ──────────────────────────────────────────────
# Sin augmentaciones — baseline limpio
train_loader_a = build_dataloader(train_df, shuffle=True)
val_loader_a   = build_dataloader(val_df)

# ── Loss / Optimizer / Scheduler ────────────────────────────
criterion_a = nn.CrossEntropyLoss()
optimizer_a = torch.optim.Adam(model_a.fc.parameters(), lr=1e-3)
scheduler_a = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_a, mode='max', patience=3, factor=0.5, verbose=True
)

# ── Entrenamiento ─────────────────────────────────────────────
model_a, history_a, best_map_a = run_experiment(
    'Exp A',
    model_a, train_loader_a, val_loader_a,
    criterion_a, optimizer_a, n_epochs=25,
    epoch_scheduler=scheduler_a, scheduler_metric='map',
)
print(f'\nMejor mAP validacion Exp A: {best_map_a:.4f}')

In [ ]:
plot_curves(history_a, 'Exp A — ResNet-50 (head only)')

---
## Experimento B — EfficientNet-B2 | Fine-tuning parcial

### Por qué EfficientNet-B2

EfficientNet aplica **compound scaling**: escala simultáneamente profundidad, ancho y resolución  
con un coeficiente único. Esto produce mejor relación accuracy/parámetros que ResNet.

EfficientNet-B2 específicamente:
- **9M parámetros** vs. 25M de ResNet-50 → 3× más compacto, `~36 MB` como `.pth`  
  (debajo del límite de GitHub sin necesidad de Git LFS).
- Accuracy en ImageNet comparable a ResNet-50 con menos parámetros → regularización implícita mayor.

### Por qué congelar los primeros 5 bloques y descongelar los últimos 3

En EfficientNet, `features[0..4]` captura edges, texturas, colores y formas simples —  
features universales que transfieren bien a cualquier dominio visual.  
`features[5..8]` captura semántica de alto nivel (específica de las 1000 clases de ImageNet)  
y necesita adaptación para distinguir 26 caras de futbolistas.

Esta división sigue la heurística de *descongelar desde arriba hacia abajo* y equilibra:
- **Adaptación**: features de alto nivel deben aprender a separar estas 26 caras.
- **Estabilidad**: descongelar todo con ~28 imágenes/clase en train causaría olvido catastrófico.

### Por qué learning rates diferenciales (AdamW)

Si aplicamos el mismo LR a capas preentrenadas y a la nueva cabeza, hay una tensión:  
la cabeza necesita un LR alto para aprender desde cero, pero los bloques preentrenados  
se destruirían con ese mismo LR.

La solución es **discriminative fine-tuning** (Howard & Ruder, ULMFiT 2018):
- `backbone_params`: `lr = 5e-5` (10× más lento que la cabeza).
- `head_params`: `lr = 5e-4`.

AdamW agrega la penalización L2 directamente sobre los pesos  
(no sobre el gradiente como Adam) → mejor regularización durante fine-tuning.

### Por qué label_smoothing = 0.1

Con ~28 imágenes/clase en train, el modelo tiende a ser sobreconfiante  
(probabilidad cercana a 1.0 para la clase correcta). Esto degrada el mAP:  
la curva precision-recall se "aplana" en valores altos de confianza.  
`label_smoothing=0.1` distribuye 10% de la masa de probabilidad a las otras 25 clases  
→ mejor calibración → mejora directa en mAP.

### Por qué CosineAnnealingLR

Baja el LR siguiendo una curva coseno desde `lr_max` hasta ~0 en `T_max=30` épocas,  
sin discontinuidades como el step decay. Complementa bien AdamW porque la penalización L2  
y la reducción suave del LR actúan juntas como regularización.

In [ ]:
set_seed()

# ── Modelo ──────────────────────────────────────────────────
model_b = models.efficientnet_b2(weights=EfficientNet_B2_Weights.IMAGENET1K_V1)

# Congelar primeros 5 bloques (stem + MBConv iniciales)
for i, block in enumerate(model_b.features):
    if i < 5:
        for param in block.parameters():
            param.requires_grad = False

# Reemplazar cabeza
in_feats_b = model_b.classifier[-1].in_features
model_b.classifier[-1] = nn.Linear(in_feats_b, NUM_CLASSES)
model_b = model_b.to(DEVICE)

n_train = sum(p.numel() for p in model_b.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model_b.parameters())
print(f'Parametros entrenables: {n_train:,} / {n_total:,}')

# ── DataLoaders ──────────────────────────────────────────────
train_loader_b = build_dataloader(train_df, use_flip=True, use_color_jitter=True, shuffle=True)
val_loader_b   = build_dataloader(val_df)

# ── Grupos de parametros con LR diferencial ──────────────────
backbone_params_b = [
    p for name, p in model_b.named_parameters()
    if p.requires_grad and 'classifier' not in name
]
head_params_b = list(model_b.classifier.parameters())

criterion_b = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer_b = torch.optim.AdamW([
    {'params': backbone_params_b, 'lr': 5e-5, 'weight_decay': 1e-4},
    {'params': head_params_b,     'lr': 5e-4, 'weight_decay': 1e-4},
])
scheduler_b = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_b, T_max=30)

# ── Entrenamiento ─────────────────────────────────────────────
model_b, history_b, best_map_b = run_experiment(
    'Exp B',
    model_b, train_loader_b, val_loader_b,
    criterion_b, optimizer_b, n_epochs=30,
    epoch_scheduler=scheduler_b,
)
print(f'\nMejor mAP validacion Exp B: {best_map_b:.4f}')

In [ ]:
plot_curves(history_b, 'Exp B — EfficientNet-B2 (fine-tuning parcial)')

---
## Experimento C — MobileNetV3-Large | Fine-tuning end-to-end

### Por qué MobileNetV3-Large

MobileNetV3-Large fue diseñado para **inferencia eficiente en dispositivos con recursos limitados**.  
Sus características clave son:
- **Inverted residuals + squeeze-and-excitation**: mejoran la capacidad semántica sin aumentar el costo computacional.
- **Hard-Swish activation**: aproximación eficiente de Swish, más rápida en hardware real.
- **5.4M parámetros, ~22 MB como `.pth`**: el modelo más liviano de los tres.

El tamaño es el factor decisivo para la **Semana 4 (despliegue en Streamlit Cloud)**:  
la plataforma tiene ~1 GB de RAM y límites de carga. Con 22 MB el modelo se carga en segundos  
y no requiere Git LFS. ResNet-50 (~100 MB) y EfficientNet-B2 (~36 MB) son alternativas válidas  
pero MobileNet favorece el despliegue sin fricciones.

### Por qué fine-tuning end-to-end

Exp A y B testean estrategias conservadoras. Exp C testa el extremo opuesto:  
*¿puede el fine-tuning completo superarlos si se regula correctamente?*

El riesgo principal es el **olvido catastrófico**: actualizaciones agresivas en los  
primeros pasos destruyen los pesos preentrenados. Se mitiga con tres mecanismos:

1. **OneCycleLR con warmup (30% de los pasos)**: el LR empieza muy bajo, sube gradualmente  
   hasta `max_lr` y luego baja. La fase de warmup evita updates grandes cuando los gradientes  
   son ruidosos al inicio.
2. **SGD + momentum + weight decay**: SGD no adapta el LR por parámetro → converge más lento  
   pero generaliza mejor en datasets pequeños (Kornblith et al. 2019).
3. **Dropout aumentado**: el clasificador original de MobileNet tiene `Dropout(0.2)`;  
   lo subimos a `Dropout(0.3)` para añadir regularización extra al entrenar todo end-to-end.

### Por qué augmentaciones más agresivas

Con end-to-end fine-tuning el modelo tiene más capacidad para memorizar el set de entrenamiento.  
Se agregan dos augmentaciones sobre las ya usadas en B:

- **RandomRotation(±10°)**: las fotos de jugadores no siempre están perfectamente alineadas  
  (tiro de cabeza, festejo). Rotaciones pequeñas dan invarianza a inclinaciones de la cabeza.
- **RandomErasing(p=0.2, scale=(0.02, 0.15))**: enmascara regiones pequeñas de la cara,  
  simulando oclusiones parciales (pelo sobre los ojos, manos, cascos).  
  Obliga al modelo a no depender de ninguna región específica.  
  Se aplica **después de `ToTensor`** para que las regiones enmascaradas queden en valor 0  
  (que equivale a la media de normalización).

### Por qué OneCycleLR en lugar de los schedulers de A y B

`OneCycleLR` implementa **super-convergence** (Smith & Topin 2018): un LR que sube y baja  
en un único ciclo puede converger más rápido y llegar a mejores mínimos que cualquier LR fijo.  
El warmup actúa como búsqueda del LR óptimo en las primeras épocas.

**Atención técnica**: `OneCycleLR` debe avanzarse **por batch**, no por época.  
Se pasa como `batch_scheduler` a `run_experiment`, que lo llama dentro de `train_one_epoch`.

In [ ]:
set_seed()

# ── Modelo ──────────────────────────────────────────────────
model_c = models.mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.IMAGENET1K_V2)

# Aumentar dropout del clasificador original (0.2 → 0.3) y reemplazar cabeza
in_feats_c = model_c.classifier[-1].in_features   # 1280
model_c.classifier[-1] = nn.Linear(in_feats_c, NUM_CLASSES)
model_c.classifier[2]  = nn.Dropout(p=0.3)        # era Dropout(0.2)
model_c = model_c.to(DEVICE)

n_train = sum(p.numel() for p in model_c.parameters() if p.requires_grad)
print(f'Parametros entrenables: {n_train:,} (todos — end-to-end)')

# ── DataLoaders (augmentaciones mas agresivas) ───────────────
train_loader_c = build_dataloader(
    train_df,
    use_flip=True, use_color_jitter=True,
    use_rotation=True, use_erasing=True,
    shuffle=True,
)
val_loader_c = build_dataloader(val_df)

# ── Loss / Optimizer / Scheduler ────────────────────────────
criterion_c = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer_c = torch.optim.SGD(
    model_c.parameters(), lr=5e-3, momentum=0.9, weight_decay=5e-4
)
# OneCycleLR avanza por batch → steps_per_epoch = len(train_loader)
scheduler_c = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_c,
    max_lr=5e-3,
    epochs=40,
    steps_per_epoch=len(train_loader_c),
    pct_start=0.3,         # 30% del tiempo en fase de warmup
    anneal_strategy='cos', # annealing coseno
)

# ── Entrenamiento ─────────────────────────────────────────────
model_c, history_c, best_map_c = run_experiment(
    'Exp C',
    model_c, train_loader_c, val_loader_c,
    criterion_c, optimizer_c, n_epochs=40,
    batch_scheduler=scheduler_c,   # OneCycleLR por batch
)
print(f'\nMejor mAP validacion Exp C: {best_map_c:.4f}')

In [ ]:
plot_curves(history_c, 'Exp C — MobileNetV3-Large (end-to-end)')

---
## 5. Comparación de los 3 experimentos

La tabla muestra los mejores valores en **validación**.  
El modelo ganador se evaluará sobre **test** en la sección siguiente  
(el conjunto de test no se toca hasta este punto).

In [ ]:
results_df = pd.DataFrame({
    'Experimento':        ['A — ResNet-50 (head)', 'B — EfficientNet-B2 (parcial)', 'C — MobileNetV3 (e2e)'],
    'Capas descongeladas':['Solo head (~53K)',     'Ultimos 3 bloques + head (~3.4M)', 'Todas (~5.4M)'],
    'Augmentaciones':     ['Ninguna',              'Flip + ColorJitter',               'Flip + CJ + Rotation + Erasing'],
    'Optimizer':          ['Adam',                 'AdamW (lr diferencial)',            'SGD + OneCycleLR'],
    'Label smoothing':    ['No',                   '0.1',                              '0.1'],
    'mAP (val)':          [round(best_map_a, 4),   round(best_map_b, 4),               round(best_map_c, 4)],
    'Acc (val)':          [round(max(history_a['val_acc']), 4),
                           round(max(history_b['val_acc']), 4),
                           round(max(history_c['val_acc']), 4)],
    'F1 (val)':           [round(max(history_a['val_f1']), 4),
                           round(max(history_b['val_f1']), 4),
                           round(max(history_c['val_f1']), 4)],
    'Tamano .pth':        ['~100 MB', '~36 MB', '~22 MB'],
})

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
print(results_df.to_string(index=False))

### Selección del ganador

El modelo ganador se selecciona por **mayor mAP en validación**.  
Si dos modelos están a menos de 0.01 de mAP, el desempate lo gana el más liviano  
(favorece el despliegue en Semana 4).

La evaluación final se realiza **únicamente sobre test**, que no se usó en ningún momento  
durante el entrenamiento ni la selección de hiperparámetros.

In [ ]:
# ── 6. Evaluacion final en test ──────────────────────────────

candidates = [
    ('A — ResNet-50',       model_a, best_map_a),
    ('B — EfficientNet-B2', model_b, best_map_b),
    ('C — MobileNetV3',     model_c, best_map_c),
]
best_name, best_model, best_val_map = max(candidates, key=lambda x: x[2])
print(f'Modelo seleccionado: {best_name}')
print(f'mAP en validacion  : {best_val_map:.4f}')

test_loader  = build_dataloader(test_df)
test_metrics = make_metrics()
_, test_m    = evaluate(best_model, test_loader, nn.CrossEntropyLoss(), test_metrics)

print(f"\n{'='*45}")
print(f'  Metricas finales en TEST — {best_name}')
print(f"{'='*45}")
print(f"  mAP (macro)  : {test_m['map']:.4f}")
print(f"  Accuracy     : {test_m['acc']:.4f}")
print(f"  F1 (macro)   : {test_m['f1']:.4f}")
print(f"{'='*45}")

In [ ]:
# ── Matriz de confusion ──────────────────────────────────────

cm_metric    = MulticlassConfusionMatrix(num_classes=NUM_CLASSES).to(DEVICE)
all_preds    = []
all_true     = []

best_model.eval()
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        logits = best_model(imgs)
        cm_metric.update(torch.softmax(logits, 1), labels)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_true.extend(labels.cpu().numpy())

cm          = cm_metric.compute().cpu().numpy()
class_names = [idx_to_label[i].replace('_', ' ').split()[-1] for i in range(NUM_CLASSES)]

fig, ax = plt.subplots(figsize=(16, 14))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(NUM_CLASSES))
ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(class_names, rotation=90, fontsize=7)
ax.set_yticklabels(class_names, fontsize=7)
ax.set_xlabel('Predicho', fontsize=11)
ax.set_ylabel('Real', fontsize=11)
ax.set_title(f'Matriz de confusion — Test | {best_name}', fontsize=12)
plt.colorbar(im, ax=ax, fraction=0.03)
plt.tight_layout()
plt.show()

# Peores 5 clases
per_class_acc = cm.diagonal() / cm.sum(axis=1).clip(min=1)
worst_5 = sorted(enumerate(per_class_acc), key=lambda x: x[1])[:5]
print('\nPeores 5 clases (accuracy por clase):')
for idx, acc in worst_5:
    print(f'  {idx_to_label[idx]:<30} {acc:.2%}')

In [ ]:
# ── Analisis de errores ──────────────────────────────────────

errors = [
    (path, true_l, pred_l)
    for path, true_l, pred_l in zip(test_df['image_path'].tolist(), all_true, all_preds)
    if true_l != pred_l
]

print(f'Errores en test: {len(errors)} / {len(test_df)} ({100*len(errors)/len(test_df):.1f}%)')

n_show = min(10, len(errors))
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
for ax, (img_path, true_l, pred_l) in zip(axes.flatten(), errors[:n_show]):
    try:
        img = Image.open(Path(FACES_DIR) / img_path).convert('RGB')
        ax.imshow(img)
    except Exception:
        ax.set_facecolor('#eee')
    real_name = idx_to_label[true_l].replace('_', ' ').split()[-1]
    pred_name = idx_to_label[pred_l].replace('_', ' ').split()[-1]
    ax.set_title(f'Real: {real_name}\nPred: {pred_name}', fontsize=8, color='darkred')
    ax.axis('off')

plt.suptitle('Ejemplos mal clasificados', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── 7. Guardado del modelo ───────────────────────────────────

save_dir  = Path('/content/dev')
save_dir.mkdir(parents=True, exist_ok=True)
save_path = save_dir / 'modelo.pth'

torch.save(best_model.state_dict(), save_path)

size_mb = save_path.stat().st_size / 1024**2
print(f'Modelo guardado: {save_path}  ({size_mb:.1f} MB)')

if size_mb > 95:
    print('Advertencia: supera 95 MB. Configurar Git LFS antes de commitear:')
    print('  git lfs install')
    print("  git lfs track 'dev/*.pth'")
    print('  git add .gitattributes dev/modelo.pth')
else:
    print('Tamano OK para GitHub sin Git LFS.')

# Descargar a local (descomentar si se desea)
# from google.colab import files
# files.download(str(save_path))